<a href="https://colab.research.google.com/github/Abhishek-S0111/Pipeline-to-Gather-Audio-Transcript-Datasets-from-Youtube/blob/new-main/Hindi_Speech_Dataset_Cleaner_and_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hindi Transcript NLP Cleaning & Dataset Preparation

This notebook automates the transformation of raw Hindi transcripts into high-quality training data for speech-to-text or fine-tuning models. It handles metadata removal, phonetic text conversion, and bulk audio acquisition.

In [29]:
import os
import time
from google.colab import drive
from google import genai
from google.colab import userdata

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Secure API Key Initialization
# Make sure to add GOOGLE_API_KEY to your Colab secrets
API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=API_KEY)

# 3. Define directory paths for text-only migration
src_base_path = "/content/drive/MyDrive/AnnamAI Tasks/Golden Database for Transcript/VzSq2eW8ZG8"
finetune_base = "/content/drive/MyDrive/AnnamAI Tasks/FineTuning_Dataset"

if not os.path.exists(src_base_path):
    raise FileNotFoundError(f"Could not find source directory at: {src_base_path}")

# 4. Scan video ID folder structures from Drive
video_ids = [f for f in os.listdir(src_base_path) if os.path.isdir(os.path.join(src_base_path, f))]
print(f"📂 Found {len(video_ids)} video folder(s) to process.\n" + "="*60)

# 5. System instructions for the Gemini text-to-text transformation
cleaning_prompt = """
You are an expert NLP Data Cleaning Engineer. Your task is to take the provided raw Hindi transcript and completely strip away all metadata to leave only the clean, raw spoken text.

Rules:
1. Remove all timestamps like [HH:MM:SS] or [MM:SS].
2. Remove all speaker tags and brackets like [मेजबान]: or [विशेषज्ञ]:.
3. Normalize irregular spaces, clear out empty line gaps, and join lines into continuous natural paragraphs. Do not alter or omit any spoken words.
4. Return ONLY the pristine Hindi text output. Do not include any conversational filler, explanations, markdown formatting, or notes.
"""

# 6. Loop and process text via Gemini with Intelligent Backoff Error Catching
for v_id in video_ids:
    print(f"🔄 Processing Transcript for Video ID: {v_id}")

    src_transcript_path = os.path.join(src_base_path, v_id, "transcript.txt")
    dest_folder = os.path.join(finetune_base, v_id)
    dest_transcript = os.path.join(dest_folder, "transcript.txt")

    os.makedirs(dest_folder, exist_ok=True)

    if os.path.exists(src_transcript_path):
        with open(src_transcript_path, "r", encoding="utf-8") as f:
            raw_transcript_text = f.read()

        max_retries = 3
        success = False

        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model='gemini-2.5-flash-lite',
                    contents=[cleaning_prompt, f"Raw Transcript to Clean:\n\n{raw_transcript_text}"]
                )
                with open(dest_transcript, "w", encoding="utf-8") as f:
                    f.write(response.text.strip())
                print("  ✓ transcript.txt successfully cleaned by Gemini and saved.")
                success = True
                break
            except Exception as e:
                if "503" in str(e) or "UNAVAILABLE" in str(e):
                    wait_time = (attempt + 1) * 5
                    print(f"  ⚠️ Server busy (503). Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"  ❌ Error for {v_id}: {e}")
                    break
    else:
        print(f"  ❌ Missing source transcript file at: {src_transcript_path}")
    print("-" * 50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Found 0 video folder(s) to process.


### Explanation for Cell `vXI3vCQBDlOI` (Initial Cleaning)
This cell performs the following actions:
1. **Connects to Storage**: Mounts your Google Drive so the script can read the transcripts.
2. **AI Setup**: Initializes the Gemini 2.5 Flash Lite model using your API key.
3. **Basic Cleaning**: Uses a prompt to tell the AI to remove timestamps (like `[00:01]`) and speaker labels (like `[Host]:`).
4. **Error Handling**: Includes a 'retry' loop. If the AI server is busy (Error 503), it waits a few seconds and tries again automatically.

## Step 1: Migration and Initial Metadata Removal
This section handles the mounting of Google Drive and performs a preliminary cleaning pass on the transcripts. It uses the Gemini 2.5 Flash Lite model to strip timestamps and basic speaker tags while preserving the core spoken Hindi text.

In [30]:
import os
import time
from google.colab import drive
from google import genai
from google.colab import userdata

# 1. Ensure Google Drive is mounted
drive.mount('/content/drive')

# 2. Secure API Key Initialization
API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=API_KEY)

# 3. Define directory paths for text-only dataset generation
src_base_path = "/content/drive/MyDrive/AnnamAI Tasks/Golden Database for Transcript/SxgaGXcQ8ZY"
finetune_base = "/content/drive/MyDrive/AnnamAI Tasks/FineTuning_Dataset"

if not os.path.exists(src_base_path):
    raise FileNotFoundError(f"Could not find source directory at: {src_base_path}")

# 4. Scan video ID folder structures from Drive
video_ids = [f for f in os.listdir(src_base_path) if os.path.isdir(os.path.join(src_base_path, f))]
print(f"📂 Found {len(video_ids)} video folder(s) to process via AI.\n" + "="*60)

# 5. Advanced System Instructions
cleaning_prompt = """
You are an expert NLP Data Cleaning Engineer. Rules:
1. REMOVE METADATA: Remove timestamps and speaker tags.
2. TOTAL SCRIPT ISOLATION: ONLY Devnagari Hindi script.
3. NUMBERS TO HINDI WORDS: Convert all digits to spoken Hindi words.
4. SYMBOLS TO HINDI WORDS: Convert symbols like % to 'प्रतिशत'.
5. ZERO WHITE GAPS: Single continuous line of text.
"""

# 6. Loop and execute text transformations via Gemini
for v_id in video_ids:
    print(f"🔄 Processing via AI for Video ID: {v_id}")
    src_transcript_path = os.path.join(src_base_path, v_id, "transcript.txt")
    dest_folder = os.path.join(finetune_base, v_id)
    dest_transcript = os.path.join(dest_folder, "transcript.txt")
    os.makedirs(dest_folder, exist_ok=True)

    if os.path.exists(src_transcript_path):
        with open(src_transcript_path, "r", encoding="utf-8") as f:
            raw_transcript_text = f.read()
        try:
            response = client.models.generate_content(
                model='gemini-3.1-flash-lite',
                contents=[cleaning_prompt, f"Raw Transcript to Clean:\n\n{raw_transcript_text}"]
            )
            with open(dest_transcript, "w", encoding="utf-8") as f:
                f.write(response.text.strip())
            print("  ✓ transcript.txt successfully processed by AI and saved.")
        except Exception as e:
            print(f"  ❌ Gemini API pipeline error for {v_id}: {e}")
    else:
        print(f"  ❌ Missing source transcript file at: {src_transcript_path}")
    print("-" * 50)

print("\n✅ AI processing pass attempted across your file structure!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Found 0 video folder(s) to process via AI.

✅ AI processing pass attempted across your file structure!


### Explanation for Cell `yFntrEIFJEnY` (Advanced Normalization)
This cell is more advanced and focuses on making the text perfect for machine learning:
1. **Hindi Only**: It instructs the AI to strip away any English words or letters.
2. **Number Conversion**: It converts digits into spoken Hindi (e.g., '5' becomes 'पाँच').
3. **Unbroken Text**: It removes all line breaks and extra spaces, turning the entire transcript into one single, long line of text. This is often required for specific fine-tuning formats.

## Step 2: Advanced Script Normalization and Phonetic Conversion
In this phase, we perform deep cleaning for model training readiness:
- **Script Isolation:** Removes all non-Devanagari characters.
- **Numerical Expansion:** Converts digits (e.g., 2024) into spoken Hindi words.
- **Symbol Handling:** Translates symbols into Hindi equivalents.
- **Flattening:** Joins all text into a single, continuous line without breaks.

In [3]:
import os
import time
from google.colab import drive
from google import genai
from google.colab import userdata

# 1. Ensure Google Drive is mounted
drive.mount('/content/drive')

# 2. Secure API Key Initialization
API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=API_KEY)

# 3. Define directory paths
src_base_path = "/content/drive/MyDrive/AnnamAI Tasks/Golden Database for Transcript"
finetune_base = "/content/drive/MyDrive/AnnamAI Tasks/FineTuning_Dataset"

# 4. Explicitly target only the remaining video IDs
video_ids = ["VzSq2eW8ZG8", "SxgaGXcQ8ZY"]

# 5. Advanced System Instructions
cleaning_prompt = """
You are an expert NLP Data Cleaning Engineer. Transform transcript into continuous training text.
1. REMOVE METADATA.
2. TOTAL SCRIPT ISOLATION: Hindi script only.
3. NUMBERS TO HINDI WORDS.
4. SYMBOLS TO HINDI WORDS.
5. ZERO WHITE GAPS: Single continuous unbroken line.
"""

# 6. Loop and execute
for v_id in video_ids:
    src_transcript_path = os.path.join(src_base_path, v_id, "transcript.txt")
    dest_folder = os.path.join(finetune_base, v_id)
    dest_transcript = os.path.join(dest_folder, "transcript.txt")
    os.makedirs(dest_folder, exist_ok=True)

    if os.path.exists(src_transcript_path):
        with open(src_transcript_path, "r", encoding="utf-8") as f:
            raw_transcript_text = f.read()
        for attempt in range(3):
            try:
                response = client.models.generate_content(
                    model='gemini-3.1-flash-lite',
                    contents=[cleaning_prompt, f"Raw Transcript to Clean:\n\n{raw_transcript_text}"]
                )
                with open(dest_transcript, "w", encoding="utf-8") as f:
                    f.write(response.text.strip())
                break
            except Exception as e:
                time.sleep(5)
print("\n✅ Targeted AI processing pass finished!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Target processing list configured for 2 specific folders.
🔄 Processing via AI for Video ID: VzSq2eW8ZG8
  🤖 Sending text data block to Gemini...
  ✓ transcript.txt successfully processed by AI and saved.
--------------------------------------------------
🔄 Processing via AI for Video ID: SxgaGXcQ8ZY
  🤖 Sending text data block to Gemini...
  ✓ transcript.txt successfully processed by AI and saved.
--------------------------------------------------

✅ Targeted AI processing pass finished!


### Explanation for Cell `HNqcu3RaRD9X` (Targeted Processing)
This cell is a safety tool. Instead of scanning all folders, it manually targets two specific Video IDs (`VzSq2eW8ZG8` and `SxgaGXcQ8ZY`). This is useful if you want to re-run the cleaning on just a few files without waiting for the entire database to be processed again.

## Step 3: Targeted Batch Processing
This specific block is used to re-process or target specific folders that may have failed during bulk processing or require updated cleaning logic using the Gemini 3.1 Flash Lite engine.

In [5]:
# 1. Install dependencies silently
!pip install -q yt-dlp

import os
import yt_dlp

# 2. Define the missing Download_Audio function
def Download_Audio(video_id):
    """
    Downloads audio from YouTube using the video ID and saves it as an MP3 file.
    """
    video_url = f"https://www.youtube.com/watch?v={video_id}"
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': f'{video_id}.%(ext)s',  # Saves the file named as the video ID
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '0',
        }],
        'quiet': True,  # Suppresses massive logs
        'no_warnings': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_url])

# 3. Define directory folder paths
src_base_path = "/content/drive/MyDrive/AnnamAI Tasks/Golden Database for Transcript"
finetune_base = "/content/drive/MyDrive/AnnamAI Tasks/FineTuning_Dataset"

if not os.path.exists(src_base_path):
    raise FileNotFoundError(f"Source directory not found at: {src_base_path}")

# 4. Extract video IDs from the source directory
video_ids = [f for f in os.listdir(src_base_path) if os.path.isdir(os.path.join(src_base_path, f))]
print(f"📂 Found {len(video_ids)} video IDs to process.\n" + "="*60)

original_dir = os.getcwd()

# 5. Loop through all IDs, create targets, and download audio
for v_id in video_ids:
    print(f"\n🔄 Processing Audio for Video ID: {v_id}")

    dest_folder = os.path.join(finetune_base, v_id)
    os.makedirs(dest_folder, exist_ok=True)

    try:
        # Change working directory so yt-dlp saves directly into the nested folder
        os.chdir(dest_folder)

        print(f"  📥 Downloading audio...")
        Download_Audio(v_id)
        print(f"  ✓ Audio successfully saved to {dest_folder}")

    except Exception as e:
        print(f"  ❌ Failed download for {v_id}: {e}")

# Reset workspace path
os.chdir(original_dir)
print("\n✅ Complete audio download pass finished!")

📂 Found 30 video IDs to process.

🔄 Processing Audio for Video ID: CEOo5d0UMvw
  📥 Downloading audio...
  ✓ Audio successfully saved to /content/drive/MyDrive/AnnamAI Tasks/FineTuning_Dataset/CEOo5d0UMvw

🔄 Processing Audio for Video ID: yKGJWYEVWvk
  📥 Downloading audio...
  ✓ Audio successfully saved to /content/drive/MyDrive/AnnamAI Tasks/FineTuning_Dataset/yKGJWYEVWvk

🔄 Processing Audio for Video ID: IYj3-fMzgY8
  📥 Downloading audio...
  ✓ Audio successfully saved to /content/drive/MyDrive/AnnamAI Tasks/FineTuning_Dataset/IYj3-fMzgY8

🔄 Processing Audio for Video ID: SxgaGXcQ8ZY
  📥 Downloading audio...
  ✓ Audio successfully saved to /content/drive/MyDrive/AnnamAI Tasks/FineTuning_Dataset/SxgaGXcQ8ZY

🔄 Processing Audio for Video ID: VzSq2eW8ZG8
  📥 Downloading audio...
  ✓ Audio successfully saved to /content/drive/MyDrive/AnnamAI Tasks/FineTuning_Dataset/VzSq2eW8ZG8

🔄 Processing Audio for Video ID: 2ELe4jT7G3Y
  📥 Downloading audio...
  ✓ Audio successfully saved to /content/

### Explanation for Cell `t9kZP7gXO3oj` (Audio Downloader)
This cell manages the media part of your dataset:
1. **Installs yt-dlp**: This is a powerful tool for downloading videos and audio from the web.
2. **Audio Extraction**: It downloads the audio from YouTube and immediately converts it to `.wav` format, which is the standard high-quality format for training speech models.
3. **Organization**: It automatically saves each audio file into its corresponding folder in your `FineTuning_Dataset` directory.

## Step 4: Bulk Audio Acquisition (yt-dlp)
To complete the dataset, this section downloads the corresponding high-quality audio files directly from YouTube. It extracts the audio as WAV files and organizes them into the same directory structure as the cleaned transcripts.